In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS oil_stock")
spark.sql("USE CATALOG oil_stock")


In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_df = spark.table("oil_stock.bronze.market_5m")

display(bronze_df)

In [0]:
silver_df = (
    bronze_df
    .select(
        F.col('datetime').alias('timestamp_utc'),
        F.col('symbol'),
        F.col('currency'),
        F.col('close').cast('double'),
        F.col('high').cast('double'),
        F.col('low').cast('double'),
        F.col('open').cast('double'),
        F.col('volume').cast('long')
    )
    .dropDuplicates(['timestamp_utc', 'symbol'])
    .filter(F.col('close').isNotNull())
)

display(silver_df)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

silver_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("oil_stock.silver.market_5m")